# 🤖 Post-Training: Improving LLMs with RLHF (DPO & GRPO)

In this notebook, we’ll show how to improve a language model using **two post-training techniques**:

### 🧠 What You’ll Learn
- What **DPO (Direct Preference Optimization)** is and how it helps models choose better answers.
- What **Generalized Reinforcement Learning with Preferences (GRPO)** is and how it helps models solve tasks  improving reasoning and performance on complex tasks (math, code, logic).
- How to train small models on **real feedback data**.
- How to **observe changes in model behavior** after fine-tuning.

### 🎯 Goal
By the end, you’ll be able to:
- Load a base model (e.g., Mistral, Qwen, Falcon, TinyLlama, etc.)
- Fine-tune it using **pairs of preferred and rejected answers**
- Try both DPO and GRPO training styles
- Save your fine-tuned models for testing or deployment

### 📦 What We’ll Use
- **Hugging Face 🤗 Transformers** to load and run models
- **TRL (Transformer Reinforcement Learning)** library by Hugging Face 🤗 for DPO and GRPO
- **A small version** of the French translated [Anthropic HH-RLHF dataset](https://huggingface.co/datasets/AIffl/french_hh_rlhf)
- **Colab GPU**, so models are small enough to run quickly

### Useful links:
- [Colab notebook](https://huhttps://colab.research.google.com/)
- [Hugging Face 🤗 DPO Trainer](https://huggingface.co/docs/trl/dpo_trainer)
- [Hugging Face 🤗 GRPO Trainer](https://huggingface.co/docs/trl/grpo_trainer)

> This notebook is interactive, friendly, and high-level. You don’t need to know deep math or theory to follow along.

# 🧑‍🎓 Student Version

This notebook contains TODOs. Fill them in before running the next sections. If you get stuck, compare with the `main` branch notebook.


In [ ]:
# Load environment variables from .env
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception as e:
    print("python-dotenv not installed. Run: pip install -r requirements.txt")


In [ ]:
# check the GPU status
!nvidia-smi

# Quick start:
1-  Clone the repository: 
```bash
git clone https://github.com/BounharAbdelaziz/RLHF.git
```
2- Install the dependencies:
```bash
pip install -q -r requirements.txt
```
3- Connect to Hugging Face hub to be able to download the models and datasets.
```bash
huggingface-cli login
```
Now we are ready to go!

In [ ]:
!git clone https://github.com/BounharAbdelaziz/RLHF.git

In [ ]:
!pip install -q -r RLHF/requirements.txt

In [ ]:
!hf auth login

# DPO

In [ ]:
# Imports
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from trl import (
    DPOTrainer,
    DPOConfig,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
import torch
import os
import wandb

In [ ]:
# We will use wandb.ai for logging the experiments
WANDB_API_KEY = os.getenv("WANDB_API_KEY", "")
WANDB_PROJECT = os.getenv("WANDB_PROJECT", "")
WANDB_ENTITY = os.getenv("WANDB_ENTITY", "")
USE_WANDB = bool(WANDB_API_KEY)

if USE_WANDB:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    if WANDB_PROJECT:
        os.environ["WANDB_PROJECT"] = WANDB_PROJECT
    if WANDB_ENTITY:
        os.environ["WANDB_ENTITY"] = WANDB_ENTITY
    wandb.login()
else:
    print("WANDB disabled. Set WANDB_API_KEY in .env to enable logging.")

# Training dataset
DATASET_PATH = "AIffl/french_orca_dpo_pairs"  # french version of "Intel/orca_dpo_pairs"

# We limit to 2k samples for speed
LIMIT = 2_000

# SFT Model we will finetune
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
# Seed for reproducibility
SEED = 1998

MAX_PROMPT_LEN = 1024
MAX_LENGTH = MAX_PROMPT_LEN + 512

RUN_NAME = "DPO-french-orca-" + MODEL_NAME.split("/")[-1]


## Load the SFT Model and Tokenizer

We will need the tonkenizer in the data preparation step to apply the chat template.

In [ ]:
# Quantization configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load the model to finetune
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
)

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Set padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

## Data Preparation

In [ ]:
def preprocess_for_dpo(example):
    # TODO: build messages list with optional system + user
    messages = []

    # HINT: example has keys: system, question, chosen, rejected
    if example.get('system') and len(example['system'].strip()) > 0:
        # TODO: append system message
        ...

    # TODO: append user question
    ...

    # TODO: build prompt using the chat template
    prompt = ...

    # TODO: set chosen and rejected responses
    chosen = ...
    rejected = ...

    return {
        "prompt": prompt,
        "chosen": chosen,
        "rejected": rejected,
    }

# Download the training dataset
dataset = load_dataset(DATASET_PATH, split=f"train")
# shuffle and select a number of samples
dataset = dataset.shuffle(True).select(range(LIMIT))

# Save columns
original_columns = dataset.column_names

# Apply the preprocessing function
dpo_dataset = dataset.map(
    preprocess_for_dpo,
    remove_columns=original_columns,
)

# Filter out examples that are too long
def filter_length(example):
    prompt_length = len(tokenizer.encode(example['prompt']))
    chosen_length = len(tokenizer.encode(example['chosen']))
    rejected_length = len(tokenizer.encode(example['rejected']))

    return (prompt_length + max(chosen_length, rejected_length)) < MAX_LENGTH

dpo_dataset = dpo_dataset.filter(filter_length)

print(f"Dataset size after filtering: {len(dpo_dataset)}")


## Model Training

In [ ]:
# LoRA configuration - targeting the correct modules for Qwen2.5
# TODO: fill target_modules with the attention + MLP projection layers

target_modules = [
    # TODO: add Qwen2.5 modules here
]

peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules,
    modules_to_save=None,
)

# Apply LoRA to the model
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# Training configuration
training_args = DPOConfig(
    beta=0.1,  # DPO temperature parameter
    learning_rate=5e-6,  # Increased learning rate
    max_prompt_length=MAX_PROMPT_LEN,
    max_length=MAX_LENGTH,
    per_device_train_batch_size=1,  # Reduced for memory
    gradient_accumulation_steps=4,  # Increased to maintain effective batch size of 4 (1*4)
    num_train_epochs=1,
    max_grad_norm=1.0,
    logging_steps=1,
    save_steps=100,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",  # More memory efficient
    warmup_ratio=0.03, # 3% of the steps will be just a warmup
    save_strategy="steps",
    output_dir="./dpo_model",
    report_to="wandb" if USE_WANDB else "none",
    run_name=RUN_NAME,
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    fp16=True,  # Enable mixed precision
)

# Initialize the trainer - Note: no ref_model needed when using peft_config
trainer = DPOTrainer(
    model=model,
    args=training_args,
    # peft_config=peft_config,  # This automatically handles reference model
    processing_class=tokenizer,
    train_dataset=dpo_dataset,
)

# Print a sample to verify preprocessing
print("Sample from dataset:")
print(f"Prompt: {dpo_dataset[0]['prompt']}")
print(f"Chosen: {dpo_dataset[0]['chosen']}")
print(f"Rejected: {dpo_dataset[0]['rejected']}")

# Train
trainer.train()


In [ ]:
# merge LoRA adapters with the base model

from utils import merge_and_save

merge_and_save(
    base_model_name="Qwen/Qwen2.5-0.5B-Instruct",
    adapter_path="dpo_model/checkpoint-489", # final checkpoint
    output_path="dpo_model/final_merged_dpo_model",
    push_to_hub=True,  # Set to True if you want to push to Hugging Face Hub
    hub_model_name="BounharAbdelaziz/Qwen2.5-0.5B-DPO-French-Orca"
)

## Model Testing

We will test the DPO model via chat-app

In [ ]:
from chat_app import launch_chat_app

launch_chat_app()

# GRPO

In [ ]:
import torch
import re
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    prepare_model_for_kbit_training,
)
from trl import (
    GRPOConfig,
    GRPOTrainer,
)
import os
import wandb

In [ ]:
# We will use wandb.ai for logging the experiments
WANDB_API_KEY = os.getenv("WANDB_API_KEY", "")
WANDB_PROJECT = os.getenv("WANDB_PROJECT", "")
WANDB_ENTITY = os.getenv("WANDB_ENTITY", "")
USE_WANDB = bool(WANDB_API_KEY)

if USE_WANDB:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    if WANDB_PROJECT:
        os.environ["WANDB_PROJECT"] = WANDB_PROJECT
    if WANDB_ENTITY:
        os.environ["WANDB_ENTITY"] = WANDB_ENTITY
    wandb.login()
else:
    print("WANDB disabled. Set WANDB_API_KEY in .env to enable logging.")

# Training dataset
DATASET_PATH = "openai/gsm8k"  # english math dataset

# We limit to 100 samples for speed
LIMIT = 200

# SFT Model we will finetune using RL (GRPO)
# MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
# Seed for reproducibility
SEED = 1998

USE_LORA = True

if MODEL_NAME == "Qwen/Qwen2.5-0.5B-Instruct":
  # no need to quantize the 0.5B. Already small enough and can be efficiently trained.
  USE_QUANT = False
else:
  USE_QUANT = True


lora_alpha = 128
lora_r = 64
lora_dropout = 0.1

MAX_PROMPT_LEN = 128
# we usually want a much larger max-length in RL training for performance. Here it is small just for speed/illustration.
MAX_LENGTH = MAX_PROMPT_LEN + 128

RUN_NAME = "GRPO-GSM8K-limit-" + str(LIMIT) + "-" + MODEL_NAME.split("/")[-1]


## Load the SFT Model and Tokenizer

In [ ]:
# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Quantization configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load the model to finetune
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    quantization_config=quantization_config if USE_QUANT else None,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False

# Prepare model for k-bit training
if USE_QUANT:
  model = prepare_model_for_kbit_training(model)

# Add padding token if not exists
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    
if USE_LORA:
  # Configure LoRA
  lora_config = LoraConfig(
      r=lora_r,  # Rank of adaptation
      lora_alpha=lora_alpha,  # LoRA scaling parameter
      target_modules=[
          "q_proj",
          "k_proj",
          "v_proj",
          "o_proj",
          "gate_proj",
          "up_proj",
          "down_proj",
      ],  # Target modules for Qwen2.5 architecture
      lora_dropout=lora_dropout,  # LoRA dropout
      bias="none",  # Bias type
      task_type=TaskType.CAUSAL_LM,  # Task type
  )

  # Apply LoRA to the model
  model = get_peft_model(model, lora_config)

  # Print trainable parameters
  model.print_trainable_parameters()

## Data Preparation

In [ ]:
# Load GSM8K dataset
dataset = load_dataset(DATASET_PATH, "main", split=f"train[:{LIMIT}]")  # Small subset for demo

def format_prompt(example):
    # TODO: format the question with a chain-of-thought hint
    # HINT: "Question: {question}\nOkay; let's think step by step before outputing an answer.\n"
    return ...

# Prepare dataset for GRPO
def prepare_dataset(dataset):
    prompts = []
    for example in dataset:
        prompt = format_prompt(example)
        prompts.append(prompt)

    # Create a proper Dataset object
    return Dataset.from_dict({"prompt": prompts})

train_dataset = prepare_dataset(dataset)


## Reward Function Design

In [ ]:
def extract_answer(text):
    # TODO: implement regex extraction for numbers
    # HINT: patterns like "#### 42" or "The answer is 42"
    return None

# Reward function design
def reward_function(prediction, ground_truth):
    # Simple reward function for math problems
    predicted_answer = extract_answer(prediction)

    if predicted_answer is None:
        return -1.0  # Penalty for no answer

    try:
        correct_num = float(ground_truth)
        if abs(predicted_answer - correct_num) < 1e-6:
            return 1.0  # Correct answer
        else:
            return -0.85  # Wrong answer
    except:
        return -1.0

# Define reward function for trainer with correct signature
def compute_reward(prompts, completions, completion_ids=None, **kwargs):
    # Compute rewards for generated completions
    rewards = []

    for i, (prompt, completion) in enumerate(zip(prompts, completions)):
        # Extract the question from the prompt
        # The prompt format is "Question: {question}
Okay; let's think step by step before outputing an answer.
"
        question_match = re.search(r"Question: (.*?)
Okay; let's think step by step before outputing an answer.
", prompt)
        if question_match:
            question = question_match.group(1)

            # Find the corresponding example in the dataset
            ground_truth = None
            for example in dataset:
                if example['question'] == question:
                    ground_truth = example['answer'].split('####')[-1].strip()
                    break

            if ground_truth is not None:
                reward = reward_function(completion, ground_truth)
                rewards.append(reward)
            else:
                rewards.append(-1.0)  # No matching ground truth found
        else:
            rewards.append(-1.0)  # Could not extract question from prompt

    return rewards


## Model Training

In [ ]:
# GRPO Configuration
grpo_config = GRPOConfig(
    output_dir="./grpo_model",
    learning_rate=1e-5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    max_prompt_length=MAX_PROMPT_LEN,
    max_completion_length=MAX_LENGTH,
    num_generations=2, # The effective train batch size must be evenly divisible by the number of generations per prompt
    beta=0,
    epsilon=0.28,
    temperature=1,
    logging_steps=1,
    save_steps=20,
    save_total_limit=3,
    # load_best_model_at_end=True,
    # metric_for_best_model="reward",
    # greater_is_better=True,
    run_name=RUN_NAME,
    report_to="wandb" if USE_WANDB else "none",
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",  # More memory efficient
    warmup_ratio=0.03, # 3% of the steps will be just a warmup
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    fp16=True,  # Enable mixed precision
)

# Initialize trainer
trainer = GRPOTrainer(
    model=model,
    reward_funcs=compute_reward,
    args=grpo_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

# Training
print("Starting GRPO training...")
trainer.train()
print("Training completed and model saved!")

## Model Testing

In [ ]:
# Test the trained model
def test_model(question):
    prompt = f"Question: {question}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=inputs['input_ids'].shape[1] + 100,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response[len(prompt):]

# Example test
test_question = "Janet's ducks lay 16 eggs per day. She eats 3 for breakfast every morning and bakes muffins for her friends every day with 4. How many eggs does she have left?"
result = test_model(test_question)
print(f"\nTest Question: {test_question}")
print(f"Model Response: {result}")

In [ ]:
from chat_app import launch_chat_app

launch_chat_app()